# 03 · Temporal Model Comparison and Calibration

Logistic Regression, Random Forest, LightGBM, XGBoost and CatBoost share the same chronological partitions
and original transaction features. Family-specific representations are fitted only on historical observations.
Each family has two candidate configurations and at most four CPU threads. Family selection uses tuning average
precision before calibration and holdout evaluation; elapsed training time is measured rather than assumed equal.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "fraud_detection").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from fraud_detection.config import Config

config = Config.load(ROOT / "configs" / "comparison.yaml")
output = Path(config.output_dir)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.2})

In [ ]:
import gc
import logging

from fraud_detection.comparison import (
    evaluate_models,
    prepare_comparison,
    refit_models,
    tune_models,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

In [ ]:
%matplotlib inline

## 1. Chronological Partitions and Shared Features

Training fits candidate estimators, tuning selects configurations and the model family, calibration estimates
probabilities and thresholds, and holdout measures final outcomes. Whole relative days remain within one partition.
All families use the same transaction rows, missingness features, amount features and fitted category vocabularies.

In [ ]:
frame, partitions = prepare_comparison(config)
splits = json.loads((output / "splits.json").read_text())
display(pd.DataFrame(splits).T)
display(
    pd.DataFrame(
        {"candidates": {name: len(trials) for name, trials in config.model_candidates.items()}}
    )
)

## 2. Candidate Tuning and Model Selection

Logistic Regression uses imputation, scaling and sparse one-hot encoding. Random Forest uses training-vocabulary
ordinal category codes; the boosting families use native categorical handling. Every family sees the same base
feature columns, although the resulting representations differ.

Boosting early stopping uses each library's precision-recall metric. Candidate and family selection consistently
recomputes sklearn average precision on the same tuning labels. No holdout metric contributes to selection.

In [ ]:
plans = tune_models(frame, partitions, config)
candidates = pd.read_csv(output / "candidate_scores.csv")
selection = json.loads((output / "selection.json").read_text())
display(
    candidates[
        ["model", "candidate", "average_precision", "roc_auc", "fit_seconds", "refit_parameters"]
    ]
)
display(pd.Series(selection, name="frozen_family_selection"))
assert selection["holdout_used_for_selection"] is False

In [ ]:
family_scores = pd.DataFrame(plans).T
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
family_scores.tuning_average_precision.astype(float).plot.bar(ax=axes[0], rot=20)
family_scores.tuning_fit_seconds.astype(float).plot.bar(ax=axes[1], rot=20)
axes[0].set(title="Best Tuning Average Precision", ylabel="Average precision")
axes[1].set(title="Selected Candidate Fit Time", ylabel="Seconds")
plt.tight_layout()
plt.show()

## 3. Refit, Probability Calibration and Review Thresholds

Each family's selected configuration is refitted on training plus tuning with a fresh shared feature builder.
The same later calibration period supplies a separate sigmoid mapping and review threshold for each estimator.
Threshold selection maximizes expected savings while enforcing the cumulative daily review quota.
Calibration results may be optimistic because both the mapping and threshold are estimated there.

In [ ]:
fitted = refit_models(frame, partitions, config, plans)
display(
    pd.DataFrame(fitted).T[
        [
            "fit_rows",
            "input_features",
            "fit_seconds",
            "calibration_seconds",
            "threshold",
            "calibration_slope",
            "encoding",
        ]
    ]
)

## 4. Frozen Holdout Evaluation

Each frozen artifact is evaluated on identical holdout transactions. Ranking quality, calibrated probability quality,
actual review rates, review precision and recall, expected costs and prediction latency describe different tradeoffs.
The selected artifact remains the family chosen on tuning, regardless of observed holdout rankings.

In [ ]:
comparison = evaluate_models(frame, partitions, config, fitted)
display(
    comparison[
        [
            "model",
            "selected_on_tuning",
            "tuning_average_precision",
            "average_precision",
            "roc_auc",
            "brier_score",
            "log_loss",
            "review_rate",
            "review_precision",
            "review_recall",
            "expected_savings",
            "fit_seconds",
            "predict_ms_per_1000",
        ]
    ]
)
assert comparison.selected_on_tuning.sum() == 1
assert (comparison.review_rate <= config.review_rate).all()
assert json.loads((output / "selection.json").read_text())["model"] == selection["model"]
del frame
gc.collect()

In [ ]:
display(Image(filename=str(output / "comparison.png")))
display(Image(filename=str(output / "precision_recall.png")))

## Interpretation

Equal candidate counts do not imply equal wall-clock budgets or exhaustive optimization. Encoding differences
are part of each estimator pipeline and are reported explicitly. Feature importance has a different definition
for each family, so its magnitudes are not directly comparable.

The holdout period was also used in the earlier LightGBM experiment. These results provide a comparative research
evaluation rather than evidence from a newly observed production period. The root artifact, reference distribution
and threshold belong to the family selected before holdout evaluation.